# Final model, saved for reuse — and chicory distribution maps

Two deliverables.

**1. A trained model saved to disk.** Fitted on all 11 levels, exported with everything needed to apply it
to a future scan without refitting.

**2. Chicory concentration maps.** The model applied across a whole dish, producing an image of how
chicory is distributed over the powder surface rather than a single number.

## Why the maps matter for your presentation

The published work on this problem uses **point spectroscopy** — one averaged spectrum per sample, no
spatial information at all. Harohally (2021) and Coqueiro (2024) both work that way. Forchetti (2020) had
imaging but over a 4 mm² window.

You have full-dish hyperspectral images at sub-millimetre sampling. That lets you show three things none
of those papers can:

- **where** the chicory is, not just how much;
- **mixing homogeneity** as a measured quantity — relevant to industrial blending, and not reported in any
  of the three;
- that a mean composition can hide a badly mixed product.

That is your novelty. The accuracy is comparable to the published FT-NIR work; the spatial information is
not available to it at all.

## One thing to be careful about in the write-up

The batch-3 samples were used for **external validation** in the previous notebook. Folding them into
calibration here means the final model no longer has an independent test set — a sample cannot validate a
model it helped train.

Present it as: method validated externally at RMSEP 1.16 %, final model refitted on everything for
deployment. Do not present the combined model's cross-validated figure as external validation.

## 1. Setup and training

In [ ]:
import os
import json, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "figure.facecolor": "white",
                     "axes.grid": True, "grid.alpha": .3})

# ------------------------------- EDIT -------------------------------
BASE    = Path(os.environ.get("COFFEE_DATA", "."))
OLD_CSV = BASE / "samples_out" / "training_samples.csv"
NEW_CSV = BASE / "batch3_out"  / "training_samples.csv"
OUT     = Path("./final_model")

WL_MIN, WL_MAX = 1000.0, 2450.0
TRIM_PCT, K_UNIT, N_COMP = 10, 8, 6
# ---------------------------------------------------------------------

OUT.mkdir(exist_ok=True, parents=True)
CAL_LABELS = ["RB100", "RB[95:5]", "RB[90:10]", "RB[85:15]", "RB[80:20]", "RB[75:25]",
              "RB[70:30]", "RB[65:35]", "RB[60:40]", "RB[55:45]", "RB[50:50]"]
print("old:", OLD_CSV.exists(), "| new:", NEW_CSV.exists())

In [ ]:
def load(path, tag=None):
    d = pd.read_csv(path)
    d = d[d.label != "AB100"].reset_index(drop=True)
    if tag:
        for c in ("scan", "group", "plate"): d[c] = d[c].astype(str) + tag
    return d

OLD, NEW = load(OLD_CSV), load(NEW_CSV, "_b3")
COMB = pd.concat([OLD, NEW], ignore_index=True)

wc = [c for c in COMB.columns if c.replace(".", "", 1).isdigit()]
wl_all = np.array([float(c) for c in wc])
BANDMASK = (wl_all >= WL_MIN) & (wl_all <= WL_MAX)
WL = wl_all[BANDMASK]
COLS = [c for c, k in zip(wc, BANDMASK) if k]

def preprocess(R):
    # absorbance, then remove each spectrum's own offset.
    # A multiplicative brightness change becomes an additive offset in log space,
    # and subtracting the row mean deletes it exactly -- which is what makes this
    # transfer across sessions, packing density and sample height.
    R = np.clip(np.asarray(R, float), 1e-3, None)
    Z = np.log10(1.0 / R)
    return Z - Z.mean(-1, keepdims=True)

Z = preprocess(COMB[COLS].values)
print(f"{len(COMB)} patches | {len(WL)} bands ({WL[0]:.0f}-{WL[-1]:.0f} nm) | "
      f"{COMB.group.nunique()} plates")

In [ ]:
# trim within each plate, then build spatially contiguous super-patches
cal = (COMB.role == "calibration").values
pre = PLSRegression(n_components=4).fit(Z[cal], COMB.chicory_pct.values[cal])
mu0 = Z[cal].mean(0)
T0 = (Z - mu0) @ pre.x_rotations_
q = np.sum(((Z - mu0) - T0 @ pre.x_loadings_.T) ** 2, 1)
keep = pd.Series(q).groupby(COMB.group.values).rank(pct=True).values <= (1 - TRIM_PCT / 100)

idx = []
for g in pd.unique(COMB.group.values):
    ii = np.where((COMB.group.values == g) & keep)[0]
    if len(ii) < K_UNIT: continue
    n = max(1, len(ii) // K_UNIT)
    lab = KMeans(n_clusters=n, n_init=5, random_state=0).fit_predict(
        COMB.loc[ii, ["patch_row", "patch_col"]].values.astype(float))
    idx += [ii[lab == c] for c in range(n) if (lab == c).sum() > 0]
ZU = np.array([Z[i].mean(0) for i in idx])
UM = COMB.iloc[[i[0] for i in idx]].reset_index(drop=True)

c = UM.label.isin(CAL_LABELS).values
Zc, y, g = ZU[c], UM.chicory_pct.values[c], UM.group.values[c]
MODEL = PLSRegression(n_components=N_COMP).fit(Zc, y)

p = np.full(len(y), np.nan)
for tr, te in LeaveOneGroupOut().split(Zc, y, groups=y):
    p[te] = PLSRegression(n_components=N_COMP).fit(Zc[tr], y[tr]).predict(Zc[te]).ravel()
plate = pd.DataFrame(dict(g=g, y=y, p=p)).groupby(["g", "y"], as_index=False)["p"].mean()
RMSECV_UNIT  = float(np.sqrt(mean_squared_error(y, p)))
RMSECV_PLATE = float(np.sqrt(mean_squared_error(plate.y, plate.p)))

MU = Zc.mean(0)
def Qres(S):
    S = np.atleast_2d(S); T = (S - MU) @ MODEL.x_rotations_
    return np.sum(((S - MU) - T @ MODEL.x_loadings_.T) ** 2, 1)
QREF = float(np.median(Qres(ZU[(UM.label == "RB100").values])))

print(f"FINAL MODEL  {N_COMP} LV, {K_UNIT} patches/unit")
print(f"  levels {sorted(set(y))}")
print(f"  {len(y)} units from {len(set(g))} plates")
print(f"  unit RMSECV  {RMSECV_UNIT:.2f} %   plate RMSECV {RMSECV_PLATE:.2f} %   "
      f"R2 {r2_score(y, p):.4f}")
print("\n  NOTE: this is cross-validation, not external validation. The external figure is")
print("  RMSEP 1.16 % from the previous notebook, measured before batch 3 joined calibration.")

## 2. Saving the model

Everything needed to apply it to a future scan: the fitted PLS, the band list, the preprocessing
parameters, the calibration centre and the Q reference. A future sample goes through exactly the same path
or the numbers will not be comparable.

In [ ]:
BUNDLE = dict(model=MODEL, wavelengths=WL, band_columns=COLS,
              wl_min=WL_MIN, wl_max=WL_MAX, n_comp=N_COMP, k_unit=K_UNIT, trim_pct=TRIM_PCT,
              preprocessing="absorbance log10(1/R) then per-spectrum offset removal",
              cal_mean=MU, q_ref=QREF, levels=sorted(map(float, set(y))),
              rmsecv_unit=RMSECV_UNIT, rmsecv_plate=RMSECV_PLATE,
              external_rmsep=1.16, validity_chicory=[0, 50], validity_robusta=[50, 100])
with open(OUT / "chicory_model.pkl", "wb") as f:
    pickle.dump(BUNDLE, f)

meta = {k: (v if isinstance(v, (int, float, str, list)) else str(type(v)))
        for k, v in BUNDLE.items()}
meta["wavelengths"] = [float(WL[0]), float(WL[-1])]
meta["n_bands"] = int(len(WL))
(OUT / "model_info.json").write_text(json.dumps(meta, indent=2))
print("saved", OUT / "chicory_model.pkl")
print(json.dumps({k: meta[k] for k in
                  ["preprocessing", "n_comp", "k_unit", "n_bands", "levels",
                   "rmsecv_plate", "external_rmsep", "validity_robusta"]}, indent=2))

## 3. Chicory distribution maps

The model is applied across a whole dish with a sliding window, giving one prediction per window position
instead of one per dish.

**Window size matters and is not free to choose.** The model was trained on super-patches of about
8 × (12 × 12) ≈ 1150 pixels, so the window is set to ~34 × 34 pixels to match that scale. A smaller window
would be noisier than anything the model was calibrated on and the map would show measurement noise rather
than composition. The stride is smaller than the window, so the map is smoothed — neighbouring cells
overlap and are not independent.

Reading it: uniform colour means well mixed; patches and streaks mean poorly mixed. The dish mean can look
correct while the distribution is badly uneven, and that is precisely what a point measurement cannot
show you.

In [ ]:
ENVI_DTYPES = {1:"u1", 2:"i2", 3:"i4", 4:"f4", 5:"f8", 12:"u2", 13:"u4"}

def parse_hdr(p):
    import re
    txt = Path(p).read_text(errors="ignore").replace("\r", "")
    body = txt.split("\n", 1)[1] if "\n" in txt else ""
    out, pos = {}, 0
    pat = re.compile(r"^\s*([\w\s]+?)\s*=\s*(\{.*?\}|[^\n]*)", re.S | re.M)
    while pos < len(body):
        m = pat.search(body, pos)
        if not m: break
        k, v = m.group(1).strip().lower(), m.group(2).strip()
        if v.startswith("{") and not v.endswith("}"):
            e = body.find("}", m.end(2)); v = body[m.start(2):e+1]; pos = e + 1
        else: pos = m.end(2)
        out[k] = v
    for k in ["samples", "lines", "bands", "header offset", "data type", "byte order"]:
        if k in out: out[k] = int(float(out[k]))
    if "wavelength" in out:
        out["wavelength"] = np.array([float(x) for x in
            out["wavelength"].strip("{}").replace("\n", " ").split(",") if x.strip()])
    out["interleave"] = out.get("interleave", "bsq").strip().lower()
    return out

def open_cube(hdr):
    h = parse_hdr(hdr); stem = Path(hdr).with_suffix("")
    path = next((c for e in (".img", ".raw", ".hyspex", ".dat", "")
                 if (c := stem.with_suffix(e)).exists() and c != Path(hdr)), None)
    L, S, B = h["lines"], h["samples"], h["bands"]
    dtc = ENVI_DTYPES[h["data type"]]
    nb = L * S * B * np.dtype(dtc).itemsize
    fs = path.stat().st_size; dec = h.get("header offset", 0)
    off = dec if fs == dec + nb else (0 if fs == nb else fs - nb)
    dt = np.dtype(dtc).newbyteorder("<" if h.get("byte order", 0) == 0 else ">")
    shp = {"bil": (L, B, S), "bsq": (B, L, S), "bip": (L, S, B)}[h["interleave"]]
    mm = np.memmap(path, dtype=dt, mode="r", offset=off, shape=shp)
    arr = {"bil": mm.transpose(0, 2, 1), "bsq": mm.transpose(1, 2, 0), "bip": mm}[h["interleave"]]
    return arr, h["wavelength"], h

def load_panel_white(txt, wl):
    rows = []
    for line in Path(txt).read_text(errors="ignore").replace("\r", "").split("\n"):
        q = line.strip().split("\t")
        if len(q) >= 2:
            try: rows.append([float(v) for v in q[:5]])
            except ValueError: continue
    a = np.array(rows)
    return np.interp(wl, a[:, 0], a[:, 1])

In [ ]:
# ------------------------------- EDIT -------------------------------
CUBE_HDR  = BASE / "batch3_rad" / "<<< a .hdr filename >>>"
PANEL_TXT = Path("./SRT-MS-050__1_.txt")
ROI_JSON  = BASE / "batch3_out" / "rois.json"    # written when you drew the ROIs
PLATE     = "RB[55:45]"                          # which dish to map
WINDOW, STRIDE = 34, 10
# ---------------------------------------------------------------------

def chicory_map(cube_hdr, roi_json, plate, panel_txt, window=WINDOW, stride=STRIDE):
    arr, wl_cube, h = open_cube(cube_hdr)
    # the CSV column names are ROUNDED wavelengths while the cube carries exact ones,
    # so build the mask from the cube and compare on the rounded values
    cube_mask = (wl_cube >= WL_MIN) & (wl_cube <= WL_MAX)
    assert cube_mask.sum() == WL.size, (
        f"band count differs: cube {cube_mask.sum()} vs training {WL.size}")
    assert np.allclose(np.round(wl_cube[cube_mask]), np.round(WL)), \
        "band centres differ from the training data"
    rois = json.loads(Path(roi_json).read_text())
    key = next(k for k in rois if Path(cube_hdr).stem.startswith(k) or k in Path(cube_hdr).stem)
    shapes = rois[key]
    Rw = load_panel_white(panel_txt, wl_cube)

    # per-column white reference from the drawn strip
    w = shapes["WHITE"]
    r0, r1 = int(w["cy"] - w["ay"]), int(w["cy"] + w["ay"])
    c0, c1 = int(w["cx"] - w["ax"]), int(w["cx"] + w["ax"])
    W = np.full((h["samples"], wl_cube.size), np.nan)
    W[c0:c1] = np.nanmedian(np.array(arr[r0:r1, c0:c1, :], dtype=np.float32), axis=0)
    good = np.isfinite(W).all(1)
    for j in range(W.shape[1]):
        W[~good, j] = np.interp(np.where(~good)[0], np.where(good)[0], W[good, j])

    # the plate
    s = shapes[plate]
    import cv2
    H, S = h["lines"], h["samples"]
    m = np.zeros((H, S), np.uint8)
    if s["kind"] == "ellipse":
        cv2.ellipse(m, (int(s["cx"]), int(s["cy"])), (int(s["ax"]), int(s["ay"])),
                    s.get("angle", 0), 0, 360, 1, -1)
    elif s["kind"] == "poly":
        cv2.fillPoly(m, [np.array([[int(a), int(b)] for a, b in s["pts"]], np.int32)], 1)
    else:
        cv2.rectangle(m, (int(s["cx"]-s["ax"]), int(s["cy"]-s["ay"])),
                      (int(s["cx"]+s["ax"]), int(s["cy"]+s["ay"])), 1, -1)
    m = cv2.erode(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))).astype(bool)

    ys, xs = np.where(m)
    R0, R1, C0, C1 = ys.min(), ys.max()+1, xs.min(), xs.max()+1
    L = np.array(arr[R0:R1, C0:C1, :], dtype=np.float32)
    refl = L / np.maximum(W[C0:C1, :][None, :, :], 1e-9) * Rw[None, None, :]
    refl[(refl < -.05) | (refl > 1.6)] = np.nan
    refl = refl[..., cube_mask]
    sub = m[R0:R1, C0:C1]

    rr = list(range(0, refl.shape[0]-window+1, stride))
    ccp = list(range(0, refl.shape[1]-window+1, stride))
    grid = np.full((len(rr), len(ccp)), np.nan)
    for a, i in enumerate(rr):
        for b, j in enumerate(ccp):
            mm_ = sub[i:i+window, j:j+window]
            if mm_.mean() < 0.95: continue
            sp = np.nanmean(refl[i:i+window, j:j+window, :][mm_], axis=0)
            if np.all(np.isfinite(sp)):
                grid[a, b] = float(MODEL.predict(preprocess(sp).reshape(1, -1)).ravel()[0])
    return grid, refl, sub, (R0, C0)

print("Set CUBE_HDR and PLATE above, then run the next cell.")
print(f"window {WINDOW}x{WINDOW} px matches the ~{K_UNIT*12*12} px training unit; stride {STRIDE}")

In [ ]:
GRID, REFL, SUBMASK, ORIGIN = chicory_map(CUBE_HDR, ROI_JSON, PLATE, PANEL_TXT)
vals = GRID[np.isfinite(GRID)]
true_ch = float(COMB.loc[COMB.label == PLATE, "chicory_pct"].iloc[0])

print(f"{PLATE}   true chicory {true_ch:.0f} %")
print(f"  map cells      {vals.size}")
print(f"  mean           {vals.mean():.2f} %   (error {vals.mean()-true_ch:+.2f})")
print(f"  sd across dish {vals.std():.2f} %   <- mixing homogeneity")
print(f"  range          {vals.min():.1f} to {vals.max():.1f} %")
print(f"  CV             {100*vals.std()/vals.mean():.1f} %")

cmap = LinearSegmentedColormap.from_list("ch", ["#2c4a7c", "#4e9ac4", "#f4e04d", "#e8743b"])
fig, axs = plt.subplots(1, 3, figsize=(15, 4.4))
j = int(np.argmin(np.abs(WL - 1650)))
g_ = np.where(SUBMASK, REFL[..., j], np.nan)
axs[0].imshow(g_, cmap="gray"); axs[0].set_title(f"{PLATE}\nreflectance at 1650 nm")
axs[0].axis("off")

im = axs[1].imshow(GRID, cmap=cmap, vmin=max(0, true_ch-12), vmax=true_ch+12,
                   interpolation="bilinear")
plt.colorbar(im, ax=axs[1], label="chicory (%)")
axs[1].set_title(f"chicory distribution\nmean {vals.mean():.1f} %  (true {true_ch:.0f} %)")
axs[1].axis("off")

axs[2].hist(vals, bins=24, color="tab:blue", alpha=.8, edgecolor="w")
axs[2].axvline(true_ch, color="k", lw=2, label=f"true {true_ch:.0f} %")
axs[2].axvline(vals.mean(), color="crimson", ls="--", lw=2, label=f"mean {vals.mean():.1f} %")
axs[2].set_xlabel("chicory (%)"); axs[2].set_ylabel("map cells")
axs[2].set_title(f"distribution across the dish\nsd = {vals.std():.2f} %"); axs[2].legend()
plt.tight_layout()
plt.savefig(OUT / f"map_{PLATE.replace('[','').replace(']','').replace(':','-')}.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 4. Homogeneity across several dishes

Run the mapping over a few plates and compare their spread. This is a measurement nobody in the papers you
read reports, and it is a genuine result: two products can share a mean composition while one is properly
blended and the other is not.

In [ ]:
PLATES_TO_MAP = ["RB[95:5]", "RB[75:25]", "RB[55:45]"]     # edit to taste

rows, grids = [], {}
for pl_ in PLATES_TO_MAP:
    try:
        G, _, _, _ = chicory_map(CUBE_HDR, ROI_JSON, pl_, PANEL_TXT)
    except Exception as e:
        print(f"  {pl_}: skipped ({e})"); continue
    v = G[np.isfinite(G)]
    t = float(COMB.loc[COMB.label == pl_, "chicory_pct"].iloc[0])
    grids[pl_] = G
    rows.append(dict(plate=pl_, true=t, mean=round(float(v.mean()), 2),
                     error=round(float(v.mean() - t), 2), sd=round(float(v.std()), 2),
                     cv_pct=round(100 * float(v.std()) / float(v.mean()), 1), cells=int(v.size)))
H = pd.DataFrame(rows)
print(H.to_string(index=False))
H.to_csv(OUT / "homogeneity.csv", index=False)

if grids:
    fig, axs = plt.subplots(1, len(grids), figsize=(5 * len(grids), 4.2))
    axs = np.atleast_1d(axs)
    for ax, (pl_, G) in zip(axs, grids.items()):
        t = float(COMB.loc[COMB.label == pl_, "chicory_pct"].iloc[0])
        im = ax.imshow(G, cmap=cmap, vmin=max(0, t-12), vmax=t+12, interpolation="bilinear")
        plt.colorbar(im, ax=ax, label="chicory (%)")
        v = G[np.isfinite(G)]
        ax.set_title(f"{pl_}\ntrue {t:.0f} %  |  mapped {v.mean():.1f} ± {v.std():.1f} %")
        ax.axis("off")
    plt.tight_layout(); plt.savefig(OUT / "homogeneity_maps.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\nThe spread within a dish is mixing quality, not model error -- the model's own")
    print(f"uncertainty is about {RMSECV_UNIT:.1f} %. Anything beyond that is the powder.")

## 5. Applying the saved model to a future scan

```python
import pickle, numpy as np, pandas as pd
B = pickle.load(open("final_model/chicory_model.pkl", "rb"))

df = pd.read_csv("some_new_batch/training_samples.csv")     # from the extraction notebook
R  = np.clip(df[B["band_columns"]].values.astype(float), 1e-3, None)
Z  = np.log10(1/R); Z = Z - Z.mean(1, keepdims=True)

pred = B["model"].predict(Z).ravel()
df["chicory_pct_pred"] = pred
df["robusta_pct_pred"] = 100 - pred
print(df.groupby("label")[["chicory_pct_pred", "robusta_pct_pred"]].mean())
```

Two checks before quoting any number:

- **Range.** Valid for 0–50 % chicory, i.e. 50–100 % robusta. Outside that it extrapolates, and error grows
  sharply with distance past the edge.
- **Q residual.** Compare against `B["q_ref"]`. Above roughly 2× means the sample is spectrally unlike
  anything calibrated — a different roast, a different grind, or a different adulterant — and no number
  should be quoted even if it lands inside the range.

Note that the Q check catches *composition* outside the range, not *species*. An arabica-based mixture
passes it while reading 8–10 points low.